# 04 — A custom DMF

A system DMF covers one fixed question. `DUPLICATE_COUNT` counts duplicated keys, `ROW_COUNT` counts rows, `FRESHNESS` measures age. None of them answers "how many rows violate a business rule." That needs a function of your own.

`bt_positive_qty` counts the rows of a table whose `qty` is not greater than zero. A clean load returns 0. A bad row returns more than 0.

Two things about the signature, both required:

- A custom DMF takes its table as a `TABLE(...)` argument, and the columns you name there are the columns the association passes in. The function body reads that argument, not the base table.
- You call it by hand before you schedule it. `SELECT bt_positive_qty(SELECT qty FROM some_table)` runs the body once, on the warehouse, and is not billed as Data Quality Monitoring.

A custom DMF attaches and schedules the same way a system one does: `ADD DATA METRIC FUNCTION` on the object, with an expectation, under that object's `DATA_METRIC_SCHEDULE`. **It bills the same way too, so the teardown notebook drops this association along with the others.**

In [ ]:
USE ROLE BT_BUILDER_ROLE;
USE WAREHOUSE BT_WH;
USE DATABASE BT_DQ_REF;
USE SCHEMA BT_DQ_REF;

## Create it

The body returns a `NUMBER`. `COUNT_IF` counts the rows where the rule fails, so the expectation is `VALUE = 0`.

In [ ]:
CREATE OR REPLACE DATA METRIC FUNCTION bt_positive_qty(
  arg_t TABLE (arg_qty NUMBER)
)
RETURNS NUMBER
COMMENT = 'Rows whose qty is not greater than zero. Custom DMF example for the data-quality reference.'
AS
$$
  SELECT COUNT_IF(NOT (arg_qty > 0))
  FROM arg_t
$$;

In [ ]:
DESC FUNCTION bt_positive_qty(TABLE(NUMBER));

## Call it before attaching it

A direct call takes a query that projects exactly the columns in the signature. This is a plain SQL call on `BT_WH`. It is not a scheduled evaluation and it is not billed as Data Quality Monitoring.

In [ ]:
BEGIN;
UPDATE bt_batch_id SET id = id + 1;
DELETE FROM bt_orders;
INSERT INTO bt_orders (order_id, line_no, sku, qty, load_ts, batch_id)
SELECT
  1, 1, 'SKU-000001', 1, CURRENT_TIMESTAMP(), (SELECT id FROM bt_batch_id)
UNION ALL
SELECT
  1, 2, 'SKU-000001', 4, CURRENT_TIMESTAMP(), (SELECT id FROM bt_batch_id);
INSERT INTO bt_load_ledger (batch_id, batch_name, attempted_at, row_count, note)
SELECT id, 'custom_dmf_good', CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, 2, 'qty > 0'
FROM bt_batch_id;
COMMIT;

In [ ]:
SELECT bt_positive_qty(SELECT qty FROM bt_orders) AS bad_qty_rows;

In [ ]:
BEGIN;
UPDATE bt_batch_id SET id = id + 1;
INSERT INTO bt_orders (order_id, line_no, sku, qty, load_ts, batch_id)
SELECT
  2, 1, 'SKU-000002', 0, CURRENT_TIMESTAMP(), (SELECT id FROM bt_batch_id)
UNION ALL
SELECT
  2, 2, 'SKU-000002', -3, CURRENT_TIMESTAMP(), (SELECT id FROM bt_batch_id);
INSERT INTO bt_load_ledger (batch_id, batch_name, attempted_at, row_count, note)
SELECT id, 'custom_dmf_bad', CURRENT_TIMESTAMP()::TIMESTAMP_NTZ, 2, 'qty is zero and negative'
FROM bt_batch_id;
COMMIT;

In [ ]:
SELECT bt_positive_qty(SELECT qty FROM bt_orders) AS bad_qty_rows;

## Attach it

Same statement as a system DMF, and it shares the schedule already set on `bt_orders` by the monitors notebook. `TRIGGER_ON_CHANGES` means it evaluates when the table is written to.

If you are running this notebook on its own and the monitors notebook has not set a schedule, set one first:

```sql
ALTER TABLE bt_orders SET DATA_METRIC_SCHEDULE = 'TRIGGER_ON_CHANGES';
```

In [ ]:
ALTER TABLE bt_orders
  ADD DATA METRIC FUNCTION bt_positive_qty
    ON (qty)
    EXPECTATION qty_positive (VALUE = 0);

In [ ]:
SELECT metric_database_name, metric_schema_name, metric_name,
       ref_entity_name, schedule, schedule_status
FROM TABLE(INFORMATION_SCHEMA.DATA_METRIC_FUNCTION_REFERENCES(
  REF_ENTITY_NAME => 'BT_DQ_REF.BT_DQ_REF.bt_orders',
  REF_ENTITY_DOMAIN => 'TABLE'));

## What this does not do

It does not wait for the schedule and it does not read `DATA_QUALITY_MONITORING_RESULTS`. The two direct calls above are the proof: 0 on the clean rows, 2 once the zero and the negative quantity are inserted.

The association keeps billing until the teardown notebook drops it. Run that notebook when you are finished.